<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509Files/blob/main/OPIM5509_Module5_Files/notebooks/M5_2b_Word_Embeddings.ipynb)

🔴
<!-- 🎙 DAVE TALKING POINTS — M5 · 7 — Tokenize, pad, and the Embedding layer (count its parameters)
- Say the line first: M5.1 threw word order away. M5.2 KEEPS it - and step one is turning each word into a vector.
- The picture: one-hot = 10,000 columns of mostly zeros, fixed by hand. An embedding = 8 (or 100) dense numbers per word, LEARNED. Similar words end up close together - king - man + woman lands near queen.
- Two ways to get one: learn it along with your task (this video) or borrow a pretrained one (next video).
- Embedding(1000, 64) is just a lookup table: 1,000 words x 64 numbers = 64,000 parameters.
- IMDB comes pre-tokenized in Keras: keep the top 10,000 words, and pad_sequences cuts every review to its first 20 words (short ones get zeros in front).
- Input(20) -> Embedding(10000, 8) = 80,000 -> Flatten 20 x 8 = 160 -> Dense(1) = 161. Total 80,161. The Input layer is the Keras 3 way - input_length is ignored now.
- Validation accuracy peaks at 0.76 (epoch 7) from 20 words per review; best validation loss 0.498 at epoch 5. But Flatten treats each position separately: 'this movie is shit' vs 'this movie is the shit' - recurrent layers fix that in video 9.
-->

# Word embeddings: from one-hot to learned vectors to GloVe

--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

The basic ways to turn words into numbers, from the obvious to the powerful: **one-hot** (one column per word), **learned embeddings** (a short vector per word, flattened into a Dense network), and **pre-trained GloVe** vectors on raw text. The geometry of GloVe is in M5_2a; reading words *in order* is M5_2c.

Adapted from Chollet, *Deep Learning with Python*, Chapter 6, Section 1.

In [1]:
import keras
keras.__version__

'3.15.1'

In [2]:
# seed everything: same numbers on my screen and yours
# (and we re-seed right before every model build, so model 2 is as reproducible as model 1)
import keras
import tensorflow as tf

keras.utils.set_random_seed(5509)
tf.config.experimental.enable_op_determinism()

In [3]:
# The data below is downloaded from public sources (Stanford), so no Google Drive is needed.
# (Only mount Drive if you want to save your trained model there:)
# from google.colab import drive
# drive.mount('/content/drive')

![word embeddings vs. one hot encoding](https://s3.amazonaws.com/book.keras.io/img/ch6/word_embeddings.png)

There are two ways to obtain word embeddings:

* Learn word embeddings jointly with the main task you care about (e.g. document classification or sentiment prediction).
In this setup, you would start with random word vectors, then learn your word vectors in the same way that you learn the weights of a neural network.
* Load into your model word embeddings that were pre-computed using a different machine learning task than the one you are trying to solve.
These are called "pre-trained word embeddings".

Let's take a look at both.

## Learning word embeddings with the `Embedding` layer


The simplest way to associate a dense vector to a word would be to pick the vector at random. The problem with this approach is that the
resulting embedding space would have no structure: for instance, the words "accurate" and "exact" may end up with completely different
embeddings, even though they are interchangeable in most sentences. It would be very difficult for a deep neural network to make sense of
such a noisy, unstructured embedding space.

To get a bit more abstract: the geometric relationships between word vectors should reflect the semantic relationships between these words.
Word embeddings are meant to map human language into a geometric space. For instance, in a reasonable embedding space, we would expect
synonyms to be embedded into similar word vectors, and in general we would expect the geometric distance (e.g. L2 distance) between any two
word vectors to relate to the semantic distance of the associated words (words meaning very different things would be embedded to points
far away from each other, while related words would be closer). Even beyond mere distance, we may want specific __directions__ in the
embedding space to be meaningful.

[...]


In real-world word embedding spaces, common examples of meaningful geometric transformations are "gender vectors" and "plural vector". For
instance, by adding a "female vector" to the vector "king", one obtain the vector "queen". By adding a "plural vector", one obtain "kings".
Word embedding spaces typically feature thousands of such interpretable and potentially useful vectors.

Is there some "ideal" word embedding space that would perfectly map human language and could be used for any natural language processing
task? Possibly, but in any case, we have yet to compute anything of the sort. Also, there isn't such a thing as "human language", there are
many different languages and they are not isomorphic, as a language is the reflection of a specific culture and a specific context. But more
pragmatically, what makes a good word embedding space depends heavily on your task: the perfect word embedding space for an
English-language movie review sentiment analysis model may look very different from the perfect embedding space for an English-language
legal document classification model, because the importance of certain semantic relationships varies from task to task.

It is thus reasonable to __learn__ a new embedding space with every new task. Thankfully, backpropagation makes this really easy, and Keras makes it
even easier. It's just about learning the weights of a layer: the `Embedding` layer.

In [4]:
from keras.layers import Embedding

# The Embedding layer takes at least two arguments:
# the number of possible tokens, here 1000 (1 + maximum word index),
# and the dimensionality of the embeddings, here 64.
embedding_layer = Embedding(1000, 64)


The `Embedding` layer is best understood as a dictionary mapping integer indices (which stand for specific words) to dense vectors. It takes
as input integers, it looks up these integers into an internal dictionary, and it returns the associated vectors. It's effectively a dictionary lookup.


The `Embedding` layer takes as input a 2D tensor of integers, of shape `(samples, sequence_length)`, where each entry is a sequence of
integers. It can embed sequences of variable lengths, so for instance we could feed into our embedding layer above batches that could have
shapes `(32, 10)` (batch of 32 sequences of length 10) or `(64, 15)` (batch of 64 sequences of length 15). All sequences in a batch must
have the same length, though (since we need to pack them into a single tensor), so sequences that are shorter than others should be padded
with zeros, and sequences that are longer should be truncated.

This layer returns a 3D floating point tensor, of shape `(samples, sequence_length, embedding_dimensionality)`. Such a 3D tensor can then
be processed by a RNN layer or a 1D convolution layer (both will be introduced in the next sections).

When you instantiate an `Embedding` layer, its weights (its internal dictionary of token vectors) are initially random, just like with any
other layer. During training, these word vectors will be gradually adjusted via backpropagation, structuring the space into something that the
downstream model can exploit. Once fully trained, your embedding space will show a lot of structure -- a kind of structure specialized for
the specific problem you were training your model for.

Let's apply this idea to the IMDB movie review sentiment prediction task that you are already familiar with. Let's quickly prepare
the data. We will restrict the movie reviews to the top 10,000 most common words (like we did the first time we worked with this dataset),
and cut the reviews after only 20 words. Our network will simply learn 8-dimensional embeddings for each of the 10,000 words, turn the
input integer sequences (2D integer tensor) into embedded sequences (3D float tensor), flatten the tensor to 2D, and train a single `Dense`
layer on top for classification.

In [5]:
from keras.datasets import imdb
from keras import preprocessing
from keras.utils import pad_sequences

# Number of words to consider as features
max_features = 10000
# cut every review after N words - here N = maxlen = 20
# (careful: pad_sequences keeps the LAST 20 words by default, not the first 20)
maxlen = 20

# Load the data as lists of integers.
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# This turns our lists of integers
# into a 2D integer tensor of shape `(samples, maxlen)`
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

### What does the model actually see?
Keras ships IMDB already turned into numbers - every word is replaced by its rank (1 = most common). Let's turn two reviews back into words and look at the label and the 20 numbers that go into the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Keras shifts every word's rank by 3: 0 = padding, 1 = start of review, 2 = unknown word
word_index = imdb.get_word_index()
index_word = {0: '<pad>', 1: '<start>', 2: '<unk>'}
for word, rank in word_index.items():
    index_word[rank + 3] = word
label_name = {0: 'negative', 1: 'positive'}

peek_words = []          # we'll reuse these two reviews later (GloVe!)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for k in [0, 1]:
    words = []
    for number in x_train[k]:
        words.append(index_word.get(number, '?'))
    peek_words.append(words)
    print('review', k, '| label:', y_train[k], '(' + label_name[y_train[k]] + ')')
    print('   words :', ' '.join(words))
    print('   vector:', x_train[k])
    axes[k].bar(range(1, maxlen + 1), x_train[k])
    axes[k].set_xticks(range(1, maxlen + 1))
    axes[k].set_xticklabels(words, rotation=60)
    axes[k].set_ylabel('word number')
    axes[k].set_title('review ' + str(k) + ' - label ' + str(y_train[k]) + ' (' + label_name[y_train[k]] + ')')
plt.tight_layout()
plt.show()

# THIS is what the model sees: 20 integers. The bar height is just a word's rank - it means nothing yet.
# The Embedding layer is what turns each integer into a useful vector.

### Step 1: one-hot encoding - the obvious way to turn words into numbers
Give every word its own column, and put a 1 in the column of the word you see. One review of 20 words becomes a 20 x 10,000 grid of zeros with twenty 1s. It works - and you can one-hot an entire data set - but look at the size.

In [ ]:
# one-hot review 0: one row per word, one column per vocabulary word
one_hot = np.zeros((maxlen, max_features))
for slot in range(maxlen):
    one_hot[slot, x_train[0][slot]] = 1
print('one review, one-hot:', one_hot.shape, '=', one_hot.size, 'numbers -', int(one_hot.sum()), 'of them are 1s')
print('all 25,000 training reviews would be', f'{25000 * one_hot.size:,}', 'numbers')

plt.figure(figsize=(14, 4))
rows_hit, cols_hit = np.nonzero(one_hot)
plt.scatter(cols_hit, rows_hit, marker='s', s=40)
for k in range(len(rows_hit)):
    plt.annotate(peek_words[0][rows_hit[k]], (cols_hit[k], rows_hit[k]), fontsize=8, xytext=(4, -3), textcoords='offset points')
plt.gca().invert_yaxis()
plt.xlim(0, max_features)
plt.xlabel('vocabulary column (0 - 9,999)')
plt.ylabel('word slot (1 - 20)')
plt.title('review 0, one-hot: 200,000 cells, twenty 1s - everything else is 0')
plt.show()

### Step 2: one-hot a whole data set (bag of words, the M5.1 way) - a strong baseline!
Squash each review's one-hot grid into ONE row: a 1 if the word appears anywhere in the review. That's 10,000 columns per review, word order thrown away - and a plain Dense network on it is hard to beat. We use the **full** reviews here, not just 20 words.

In [ ]:
# multi-hot: one row per review, a 1 in the column of every word it uses (order thrown away)
(full_train, y_full_train), (full_test, y_full_test) = imdb.load_data(num_words=max_features)

def multi_hot(reviews):
    out = np.zeros((len(reviews), max_features), dtype='float32')
    for r in range(len(reviews)):
        for number in reviews[r]:
            out[r, number] = 1.0
    return out

X_hot_train = multi_hot(full_train)
X_hot_test = multi_hot(full_test)
print('one-hot data set:', X_hot_train.shape, '- one row per review, one column per word')

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
model.add(Input(shape=(max_features,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_hot_train, y_full_train,
                    epochs=50,
                    batch_size=512,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - one-hot (multi-hot) bag of words, full reviews')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

# the honest score: HELD-OUT test reviews
from sklearn.metrics import classification_report, f1_score
preds = (model.predict(X_hot_test, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(y_full_test, preds, target_names=['negative', 'positive']))
print('one-hot bag of words (full reviews) - test macro F1:', round(f1_score(y_full_test, preds, average='macro'), 3))

### Step 3: embeddings - a short, dense vector per word instead of 10,000 columns
The one-hot model is strong, but it's 10,000 numbers per review and it has no idea that *great* and *excellent* mean nearly the same thing. An **Embedding** layer gives every word a short vector (8 numbers here) that the model *learns* - and words used the same way end up with similar vectors. Below: 20 words per review, embedded, flattened.

In [ ]:
from keras.models import Sequential
from keras.layers import Flatten, Dense
from keras.layers import Input
from keras.callbacks import EarlyStopping

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
# We specify the maximum input length with an Input layer
# so we can later flatten the embedded inputs
model.add(Input(shape=(maxlen,)))
model.add(Embedding(10000, 8))
# After the Embedding layer,
# our activations have shape `(samples, maxlen, 8)`.

# We flatten the 3D tensor of embeddings
# into a 2D tensor of shape `(samples, maxlen * 8)`
model.add(Flatten())

# We add the classifier on top
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# stop when validation loss stops improving, and hand back the BEST epoch's weights
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(x_train, y_train,
                    epochs=50,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - learned 8-number embeddings, 20 words')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

In [ ]:
# the honest score: HELD-OUT test reviews the model never saw while training
from sklearn.metrics import classification_report, f1_score
probs = model.predict(x_test, verbose=0).ravel()
preds = (probs > 0.5).astype(int)
print(classification_report(y_test, preds, target_names=['negative', 'positive']))
print('learned embedding (20 words) - test macro F1:', round(f1_score(y_test, preds, average='macro'), 3))

### Why Flatten is a clumsy way to read text: every word is locked to its slot
The Embedding layer turns each of the 20 words into 8 numbers - a 20 x 8 grid. `Flatten` then lays those 20 rows end to end: 160 numbers. The Dense layer gives each of the 160 its own weight, so it learns *"'excellent' in slot 20"* - a completely separate thing from *"'excellent' in slot 10"*. Same word, different slot, different weight. That's tough for a dense layer, and it's exactly what recurrent layers fix (video 9).

In [ ]:
# review 0 after the Embedding layer: 20 words x 8 numbers each
rows = model.layers[0](x_train[0:1]).numpy()[0]
flat = rows.reshape(-1)                       # what Flatten hands the Dense layer: 20 x 8 = 160 numbers
print('embedded review:', rows.shape, '-> flattened:', flat.shape)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].imshow(rows.T, aspect='auto', cmap='coolwarm')
axes[0].set_xticks(range(maxlen))
axes[0].set_xticklabels(peek_words[0], rotation=60)
axes[0].set_ylabel('embedding number (1-8)')
axes[0].set_title('review 0 after the Embedding layer: one column of 8 numbers per word')
axes[1].bar(range(len(flat)), flat)
for slot in range(1, maxlen):
    axes[1].axvline(slot * 8 - 0.5, color='grey', linewidth=0.5)
axes[1].set_title('after Flatten: the same 160 numbers in one long row - word 1 is columns 1-8, word 2 is columns 9-16, ...')
plt.tight_layout()
plt.show()

In [ ]:
# the proof: score the SAME word in different slots with the Dense layer's weights
dense_weights = model.layers[2].get_weights()[0].reshape(maxlen, 8)   # 160 weights = 20 slots x 8
word_vectors = model.layers[0].get_weights()[0]                        # the learned 8-number vector for every word
for w in ['excellent', 'great', 'awful', 'boring']:
    vector = word_vectors[word_index[w] + 3]
    scores = []
    for slot in [1, 5, 10, 15, 20]:
        scores.append(round(float(vector @ dense_weights[slot - 1]), 2))
    print(f'{w:10s} push toward positive at slots 1 / 5 / 10 / 15 / 20:', scores)

# positive words push up and negative words push down - but by a DIFFERENT amount in every slot.
# the model had to learn 'excellent' 20 separate times.

### The reveal: an Embedding layer IS one-hot x a weight matrix
Multiply review 0's one-hot grid (20 x 10,000) by the Embedding layer's weight matrix (10,000 x 8) and you get exactly the 20 x 8 grid the Embedding layer produced. The layer just skips the multiplication and looks the row up - same answer, 10,000 times cheaper.

In [ ]:
embedding_matrix_learned = model.layers[0].get_weights()[0]        # 10,000 words x 8 numbers
by_hand = one_hot @ embedding_matrix_learned                       # (20 x 10,000) @ (10,000 x 8) = 20 x 8
by_layer = model.layers[0](x_train[0:1]).numpy()[0]               # what the Embedding layer gave back
print('one-hot x weights:', by_hand.shape, '| Embedding layer:', by_layer.shape)
print('identical?', np.allclose(by_hand, by_layer))

Not bad for only 20 words per review - but flattening the embedded sequence treats each word separately, slot by slot, without considering how the words relate (it would likely treat both _"this movie is shit"_ and _"this movie is the shit"_ as negative). Recurrent or 1D convolutional layers on top of the embeddings read the review as a whole - that's where we're headed.

🔴
<!-- 🎙 DAVE TALKING POINTS — M5 · 8 (cont.) — GloVe on real IMDB text - small data on purpose
- Now from RAW text: download IMDB from Stanford, read the files (in sorted order, so Colab matches), 25,000 reviews. The tokenizer finds 87,393 unique tokens; keep 10,000, pad to 300 words.
- Small data ON PURPOSE: 150 training reviews, 51 validation (the text above still says 200).
- GloVe: 400,000 words x 100 numbers -> embedding_matrix 10,000 x 100. Words GloVe doesn't know stay all zeros.
- 300 words x 100 flattened = 30,000 inputs -> Dense(32) = 960,032. summary() says 1,960,065 all trainable because it prints BEFORE set_weights + trainable=False - after the freeze, 1,000,000 are frozen.
- 300 epochs, batch 5: train accuracy 1.0, validation loss best 0.98 at epoch 6 then climbs to 1.94. Read the loss curve - textbook overfitting. Validation accuracy 0.45-0.53.
- Same model learning its own embedding: validation peaks at 0.59. Do NOT say either one won - with 51 validation reviews, one review is 2 points. Both are coin flips.
- The honest lesson: 150 reviews isn't enough for anything. Borrowing GloVe won in the tiny example; here the fix is more data - the capstone proves it (100 posts vs all of them).
-->

## Using pre-trained word embeddings


Sometimes, you have so little training data available that could never use your data alone to learn an appropriate task-specific embedding
of your vocabulary. What to do then?

Instead of learning word embeddings jointly with the problem you want to solve, you could be loading embedding vectors from a pre-computed
embedding space known to be highly structured and to exhibit useful properties -- that captures generic aspects of language structure. The
rationale behind using pre-trained word embeddings in natural language processing is very much the same as for using pre-trained convnets
in image classification: we don't have enough data available to learn truly powerful features on our own, but we expect the features that
we need to be fairly generic, i.e. common visual features or semantic features. In this case it makes sense to reuse features learned on a
different problem.

Such word embeddings are generally computed using word occurrence statistics (observations about what words co-occur in sentences or
documents), using a variety of techniques, some involving neural networks, others not. The idea of a dense, low-dimensional embedding space
for words, computed in an unsupervised way, was initially explored by Bengio et al. in the early 2000s, but it only started really taking
off in research and industry applications after the release of one of the most famous and successful word embedding scheme: the Word2Vec
algorithm, developed by Mikolov at Google in 2013. Word2Vec dimensions capture specific semantic properties, e.g. gender.

There are various pre-computed databases of word embeddings that can download and start using in a Keras `Embedding` layer. Word2Vec is one
of them. Another popular one is called "GloVe", developed by Stanford researchers in 2014. It stands for "Global Vectors for Word
Representation", and it is an embedding technique based on factorizing a matrix of word co-occurrence statistics. Its developers have made
available pre-computed embeddings for millions of English tokens, obtained from Wikipedia data or from Common Crawl data.

Let's take a look at how you can get started using GloVe embeddings in a Keras model. The same method will of course be valid for Word2Vec
embeddings or any other word embedding database that you can download. We will also use this example to refresh the text tokenization
techniques we introduced a few paragraphs ago: we will start from raw text, and work our way up.

## Putting it all together: from raw text to word embeddings


We will be using a model similar to the one we just went over -- embedding sentences in sequences of vectors, flattening them and training a
`Dense` layer on top. But we will do it using pre-trained word embeddings, and instead of using the pre-tokenized IMDB data packaged in
Keras, we will start from scratch, by downloading the original text data.

### Why download IMDB a SECOND time? (this is the pro move)
`imdb.load_data()` is a **toy**: the words already arrive as numbers, shifted by 3, with no text to look at. Handy, but a little mysterious - and no real project hands you data like that.

In production you get **raw text files**. So now we do it the real way: download the original reviews, read every file, let the **folder names be the labels** (`neg/`, `pos/`), and build the tokenizer ourselves. We train on `aclImdb/train` and keep `aclImdb/test` **held out** for the honest score at the end.

### Download the IMDB data as raw text


First, head to `http://ai.stanford.edu/~amaas/data/sentiment/` and download the raw IMDB dataset (if the URL isn't working anymore, just
Google "IMDB dataset"). Uncompress it.

Now let's collect the individual training reviews into a list of strings, one string per review, and let's also collect the review labels
(positive / negative) into a `labels` list:

In [7]:
# Download the raw IMDB review data from Stanford (the original source, ~84MB)
!wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [8]:
# Uncompress it -> creates /content/aclImdb/ with train/ and test/ (each holding pos/ and neg/)
!tar -xzf aclImdb_v1.tar.gz

In [ ]:
import os

def read_reviews(folder):
    """Read every review in folder/neg and folder/pos - the folder name IS the label."""
    texts = []
    labels = []
    for label_type in ['neg', 'pos']:
        dir_name = os.path.join(folder, label_type)
        for fname in sorted(os.listdir(dir_name)):   # sorted: Colab (Linux) and Windows list files in different orders
            if fname[-4:] == '.txt':
                f = open(os.path.join(dir_name, fname), encoding='utf-8')
                texts.append(f.read())
                f.close()
                if label_type == 'neg':
                    labels.append(0)
                else:
                    labels.append(1)
    return texts, labels

texts, labels = read_reviews('/content/aclImdb/train')               # training reviews
test_texts, test_labels = read_reviews('/content/aclImdb/test')      # HELD OUT until the very end
print(len(texts), 'training reviews,', len(test_texts), 'held-out test reviews')
print(texts[0][:300])

### Tokenize the data


Let's vectorize the texts we collected, and prepare a training and validation split.
We will merely be using the concepts we introduced earlier in this section.

Because pre-trained word embeddings are meant to be particularly useful on problems where little training data is available (otherwise,
task-specific embeddings are likely to outperform them), we will add the following twist: we restrict the training data to its first 200
samples. So we will be learning to classify movie reviews after looking at just 200 examples...


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer  # Keras 3 dropped keras.preprocessing.text; this path still works in Colab
#from keras.preprocessing.sequence import pad_sequences
import numpy as np

maxlen = 300  # cut every review after N = 300 words (pad_sequences keeps the LAST 300)
training_samples = 150  # SMALL data on purpose: train on 150 reviews
validation_samples = 51  # ...and validate on 51
max_words = 10000  # We will only consider the top 10,000 words in the dataset

# we will use training_samples and validation_samples
# later on in a few cells... for splitting data...

In [11]:
# link: https://keras.io/preprocessing/text/
# has good details on what a tokenizer does

# This class allows to vectorize a text corpus, by turning each text into either a sequence of integers
# omits common characters
tokenizer = Tokenizer(num_words=max_words)

# fit_on_texts means it learns the indices of the words
tokenizer.fit_on_texts(texts)

# the sequences are comprised of those indices
sequences = tokenizer.texts_to_sequences(texts)

word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

# in case of low word counts, we need to pad sequences
# so that they are uniform length
data = pad_sequences(sequences, maxlen=maxlen)

# our labels, which were previously stored as a list [],
# are now converted to a numpy array for modeling
labels = np.asarray(labels)
print('Shape of data tensor:', data.shape)
print('Shape of label tensor:', labels.shape)

Found 87393 unique tokens.


Shape of data tensor: (25000, 300)
Shape of label tensor: (25000,)


In [12]:
# Split the data into a training set and a validation set
# But first, shuffle the data, since we started from data
# where sample are ordered (all negative first, then all positive).
keras.utils.set_random_seed(5509)   # same shuffle every run
indices = np.arange(data.shape[0])
np.random.shuffle(indices)
data = data[indices]
labels = labels[indices]

x_train = data[:training_samples] # from 0 to (200) training samples
y_train = labels[:training_samples]
x_val = data[training_samples: training_samples + validation_samples]
y_val = labels[training_samples: training_samples + validation_samples]

# the held-out test reviews go through the SAME tokenizer (learned on train) and the same padding
x_heldout = pad_sequences(tokenizer.texts_to_sequences(test_texts), maxlen=maxlen)
y_heldout = np.asarray(test_labels)
print('train', x_train.shape, '| validation', x_val.shape, '| held-out test', x_heldout.shape)

### Download the GloVe word embeddings


Head to `https://nlp.stanford.edu/projects/glove/` (where you can learn more about the GloVe algorithm), and download the pre-computed
embeddings from 2014 English Wikipedia. It's a 822MB zip file named `glove.6B.zip`, containing 100-dimensional embedding vectors for
400,000 words (or non-word tokens). Un-zip it.

### Pre-process the embeddings


Let's parse the un-zipped file (it's a `txt` file) to build an index mapping words (as strings) to their vector representation (as number
vectors).

In [13]:
# Download the GloVe embeddings from Stanford (the original source).
# Heads up: glove.6B.zip is ~822MB, so this cell takes a few minutes.
!wget -q https://nlp.stanford.edu/data/glove.6B.zip
# Un-zip -> extracts glove.6B.100d.txt (plus 50d/200d/300d) into /content/
!unzip -q glove.6B.zip

In [14]:
import numpy as np
glove_dir = '/content/'

embeddings_index = {}
f = open(os.path.join(glove_dir, 'glove.6B.100d.txt'))
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

Found 400000 word vectors.



Now let's build an embedding matrix that we will be able to load into an `Embedding` layer. It must be a matrix of shape `(max_words,
embedding_dim)`, where each entry `i` contains the `embedding_dim`-dimensional vector for the word of index `i` in our reference word index
(built during tokenization). Note that the index `0` is not supposed to stand for any word or token -- it's a placeholder.

In [15]:
embedding_dim = 100 # this is the dimension of the file we imported
# could use 50, but need to make sure you download that file!

embedding_matrix = np.zeros((max_words, embedding_dim))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if i < max_words:
        if embedding_vector is not None:
            # Words not found in embedding index will be all-zeros.
            embedding_matrix[i] = embedding_vector

### Define a model

We will be using the same model architecture as before:

In [16]:
from keras.models import Sequential
from keras.layers import Embedding, Flatten, Dense
from keras.layers import Input

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
model.add(Input(shape=(maxlen,)))
model.add(Embedding(max_words, embedding_dim))
model.add(Flatten())
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()

# in this model, the embeddings are turned on their sides
# when they get flattened, so our 100 words with
# 100 dimensions turns into 10000 features for modeling

# if you adjust the values for embedding_dim (make it 20?)
# and max_words (make it 30?),
# this may help you better understand things...

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 300, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 30000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │       960,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,960,065 (7.48 MB)

 Trainable params: 1,960,065 (7.48 MB)

 Non-trainable params: 0 (0.00 B)

### Load the GloVe embeddings in the model


The `Embedding` layer has a single weight matrix: a 2D float matrix where each entry `i` is the word vector meant to be associated with
index `i`. Simple enough. Let's just load the GloVe matrix we prepared into our `Embedding` layer, the first layer in our model:

In [17]:
model.layers[0].set_weights([embedding_matrix])
model.layers[0].trainable = False


Additionally, we freeze the embedding layer (we set its `trainable` attribute to `False`), following the same rationale as what you are
already familiar with in the context of pre-trained convnet features: when parts of a model are pre-trained (like our `Embedding` layer),
and parts are randomly initialized (like our classifier), the pre-trained parts should not be updated during training to avoid forgetting
what they already know. The large gradient update triggered by the randomly initialized layers would be very disruptive to the already
learned features.

### Train and evaluate

Let's compile our model and train it:

In [ ]:
from keras.callbacks import EarlyStopping

model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
history = model.fit(x_train, y_train,
                    epochs=300,
                    batch_size=5,
                    validation_data=(x_val, y_val),
                    callbacks=[early_stop],
                    verbose=0)
# saves on left hand side!
model.save_weights('pre_trained_glove_model.weights.h5')

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - frozen GloVe, 150 reviews')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

In [ ]:
# the honest score: HELD-OUT test reviews the model never saw while training
from sklearn.metrics import classification_report, f1_score
probs = model.predict(x_heldout, verbose=0).ravel()
preds = (probs > 0.5).astype(int)
print(classification_report(y_heldout, preds, target_names=['negative', 'positive']))
print('frozen GloVe, 150 training reviews - test macro F1:', round(f1_score(y_heldout, preds, average='macro'), 3))

We can also try to train the same model without loading the pre-trained word embeddings and without freezing the embedding layer. In that
case, we would be learning a task-specific embedding of our input tokens, which is generally more powerful than pre-trained word embeddings
when lots of data is available. However, in our case, we have only 200 training samples. Let's try it:

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, Flatten, Dense
from keras.layers import Input

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
model.add(Input(shape=(maxlen,)))
model.add(Embedding(max_words, embedding_dim))     # NOT loaded with GloVe, NOT frozen - learned from our 150 reviews
model.add(Flatten())
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
history = model.fit(x_train, y_train,
                    epochs=300,
                    batch_size=32,
                    validation_data=(x_val, y_val),
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - embedding learned from 150 reviews')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

In [ ]:
# the honest score: HELD-OUT test reviews the model never saw while training
from sklearn.metrics import classification_report, f1_score
probs = model.predict(x_heldout, verbose=0).ravel()
preds = (probs > 0.5).astype(int)
print(classification_report(y_heldout, preds, target_names=['negative', 'positive']))
print('learned embedding, 150 training reviews - test macro F1:', round(f1_score(y_heldout, preds, average='macro'), 3))

## The held-out test: why it matters
Both models above were scored on **25,000 test reviews they never saw** - the same tokenizer, the same padding, no peeking. Validation data helped pick the epoch, so it's not a fair final score; held-out test data is. With only 150 training reviews, neither model has much to go on - the fix is more data, not a fancier model (the capstone proves it).